# **Predictive Modeling: ELO Rating System Optimization**
**Project:**  FIFA World Cup 2026 — Prediction & Simulation Engine
**Module:** Statistical Inference & Probabilistic Outcomes  
**Objective:** To implement and calibrate a customized ELO rating system. This notebook focuses on optimizing the $K$-factor and Draw Margin ($c$) through Ranked Probability Score (RPS) minimization, using a chronological burn-in strategy.

## **Table of Contents**
* **Unit 1: Theoretical Framework**
    * 1.1 The ELO Logistic Function
    * 1.2 Goal Difference Multiplier ($M(n)$)
    * 1.3 Ranked Probability Score (RPS) as a Loss Function
* **Unit 2: Data Preprocessing for Modeling**
    * 2.1 Integer Mapping & NumPy Vectorization
    * 2.2 Time-Series Sorting & Parquet Loading
* **Unit 3: Model Calibration & Optimization**
    * 3.1 Hyperparameter Tuning (L-BFGS-B)
    * 3.2 Burn-in Strategy (1872 – 2018)
* **Unit 4: Model Validation: Qatar 2022**
    * 4.1 Backtesting & Out-of-Sample Performance
* **Unit 5: Predictive Engine for 2026**
    * 5.1 Final Rating Consolidation
    * 5.2 Poisson-Based Simulation

## **Unit 1: Theoretical Framework**

### 1.1 The ELO Logistic Function

The ELO system is based on the relative strength between two teams. This strength is measured on a continuous scale and the expected results for a team is defined by a logistic function for the qualification strength difference

$$W_e = \frac{1}{10^{-d_r / 400} + 1}$$

where $d_r = R_{\text{local}} - R_{\text{visitor}} + d_{\text{home}}$, $d_{\text{home}}$ is a local advantage factor and $R$ the rescptive ELO rank value.

After finishing a match, the ELO strength is updated:

$$R_{\text{new}} = R_{\text{ond}} + K \times (W - W_e)$$

where $W$ is the result ($1$ for local win, $0.5$ for a draw and $0$ for visitor win). The $K$ parameter is used to control the maximum impact for just one match. And symmetrically, the visitor lost the points won for the local.

For their nature, the strength parameters are fitted sequentially and chronologically. The initial values are usually fixed at 1500 points.   



In [1]:
def get_expected_score(rating_a, rating_b, hfa=0):
    delta = rating_a + hfa - rating_b
    return 1 / (10**(-delta / 400) + 1)


### 1.2 Goal Difference Multiplier ($M(n)$)

The K factor can be weighted by the difference in the result, showing a bigger impact for a bigger score difference
$$
K_{modified} = K_{base} \times M(n)$$

where $M(n)$ is a function empirically calibrated
$$
M(n)= = \begin{cases} 
1 & \text{if } N\leq 1 \\ 
1.5 & \text{if } N=2 \\ 
1.75 & \text{if } N=3 \\ 
1.75+\frac{N-3}{8} & \text{if } N\geq 4 \end{cases}
$$


In [2]:
def get_k_multiplier(n):
    n = abs(n)
    if n <= 1: return 1.0
    if n == 2: return 1.5
    if n == 3: return 1.75
    return 1.75 + (n - 3) / 8.0

### 1.3 Ranked Probability Score (RPS) as a Loss Function

For obtaining the probability of each outcome (W-D-L):
$$P(\text{Local victory}) = \frac{1}{1 + 10^{-(d_R - c) / 400}}\   \   \    P(\text{Visitor victory}) = \frac{1}{1 + 10^{(d_R + c) / 400}}$$
where $c$ is a margin for a draw and then

$$P(\text{Draw}) = 1 - P(\text{Local victory}) - P(\text{Visitor victory})$$

In particular, because in a World Cup, the home effect is negligible, we will fix $d_{\text{home}=0$

For prediction evaluation, we use the Ranked Probability Score (RPS)
$$RPS = \frac{1}{r-1} \sum_{i=1}^{r-1} (P_i - O_i)^2$$
where:
- $P_i$: is the cumulative sum of predicted probabilities till category $i$.
- $O_i$: is the cumulative sum of the actual results till category $i$.
- $r$ is the total of possible results, in our case 3 (home win, draw, away win).
This metric is calculated for each predicted match.
In order to obtain the optimal values  of $k_{base}$, $d_{\text{home}}$ and $c$, we should minimize the RPS

## Unit 2: Data Preprocessing for Modeling**
### 2.1 Time-Series Sorting & Parquet Loading

We load the dataset obtained in EDA, with features 'date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country' and 'neutral' for each row wich represent a match.
It is important to have the data ordered chronologically for ELO because it is updated by match.

In [3]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from collections import Counter

df = pd.read_parquet('../data/processed/results.parquet')
df = df.sort_values('date').reset_index(drop=True)


### 2.2 Integer Mapping & NumPy Vectorization

For better performance we turn the team names in integer IDs and the dataframe in NumPy array.

In [4]:
all_teams = sorted(list(set(df['home_team']).union(set(df['away_team']))))
team_to_id = {team: i for i, team in enumerate(all_teams)}
id_to_team = {i: team for team, i in team_to_id.items()}

df['h_id'] = df['home_team'].map(team_to_id)
df['a_id'] = df['away_team'].map(team_to_id)


## Unit 3: Model Calibration & Optimization**
### 3.1 Hyperparameter Tuning (L-BFGS-B)
To adjust the parameters we perform an optimization problem to set the optimal hyperparameters $\theta = \{k_{base}, d_{home}, c\}$. The objective is to minimize the Global Ranked Probability Score (RPS) across all matches in the training set.

Optimization Components
Objective Function: We define an objective function that iterates through the historical data_array. For each match, it updates team ratings and calculates the local RPS.
Parameter Space: 
$k_{base}$: Controls how much the ratings change after a single match. 
$d_{home}$ (HFA): Quantifies the home-field advantage (Home Field Advantage factor).
$c$ (Draw Margin): Defines the threshold in the logistic distribution where a match is likely to result in a draw.

We employ the Limited-memory Broyden–Fletcher–Goldfarb–Shanno (L-BFGS-B) algorithm. This is a quasi-Newton method particularly suited for bound-constrained optimization.

In [5]:
def train_parameters(df_train, num_teams, burn_in=1000):
    # Convert dataframe to array
    # Columns: 0:h_id, 1:a_id, 2:h_score, 3:a_score
    data_array = df_train[['h_id', 'a_id', 'home_score', 'away_score']].values.astype(int)
    
    def objective(params):
        k_base, hfa, draw_margin = params
        ratings = np.full(num_teams, 1500.0)
        total_rps = 0.0
        count = 0
        
        for i in range(len(data_array)):
            h_id, a_id, h_s, a_s = data_array[i]
            
            # ELO
            delta = ratings[h_id] + hfa - ratings[a_id]
            
            # Prediction
            p_away = 1 / (1 + 10 ** ((delta + draw_margin) / 400))
            p_home = 1 / (1 + 10 ** (-(delta - draw_margin) / 400))
            p_draw = max(0.0, 1.0 - p_home - p_away)
            
            
            
            # evaluate results
            actual = 0 if h_s > a_s else 1 if h_s == a_s else 2
            e = 1.0 if actual == 0 else 0.5 if actual == 1 else 0.0
            # if the match is post burn-in, calculate RPS
            if i >= burn_in:                
                p_cum_0 = p_home
                p_cum_1 = p_home + p_draw
                e_cum_0 = 1.0 if actual == 0 else 0.0
                e_cum_1 = 1.0 if actual <= 1 else 0.0
                total_rps += ((p_cum_0 - e_cum_0)**2 + (p_cum_1 - e_cum_1)**2) / 2.0
                count += 1
            
            # Update ratings
            m_n = get_k_multiplier(h_s - a_s)
            e_home = 1 / (1 + 10 ** (-delta / 400))
            k_mod = k_base * m_n
            
            shift = k_mod * (e - e_home)
            ratings[h_id] += shift
            ratings[a_id] -= shift
            
        return total_rps / count if count > 0 else 0

    print("Beggining parameter optimization...")
    res = minimize(objective, [20.0, 40.0, 90.0], 
                   bounds=[(5, 80), (0, 150), (10, 200)], method='L-BFGS-B')
    return res.x



In [6]:
def train_parameters(df_train, num_teams, burn_in=1000):
    # Convert dataframe to array
    # Columns: 0:h_id, 1:a_id, 2:h_score, 3:a_score
    def objective_function(params, data_array, num_teams, burn_in):
        k_base, hfa, draw_margin = params
        ratings = np.full(num_teams, 1500.0)
        total_rps = 0.0
        count = 0
    
        for i in range(len(data_array)):
            h_id, a_id, h_s, a_s = data_array[i]
            delta = ratings[h_id] + hfa - ratings[a_id]
        
            p_away = 1 / (1 + 10 ** ((delta + draw_margin) / 400))
            p_home = 1 / (1 + 10 ** (-(delta - draw_margin) / 400))
            p_draw = max(0.0, 1.0 - p_home - p_away)
        
            actual = 0 if h_s > a_s else 1 if h_s == a_s else 2
        
            if i >= burn_in:
                e_cum_0 = 1.0 if actual == 0 else 0.0
                e_cum_1 = 1.0 if actual <= 1 else 0.0
                total_rps += ((p_home - e_cum_0)**2 + (p_home + p_draw - e_cum_1)**2) / 2.0
                count += 1
        
            # Update ratings
            e = 1.0 if actual == 0 else 0.5 if actual == 1 else 0.0
            e_home = 1 / (1 + 10 ** (-delta / 400))
            shift = (k_base * get_k_multiplier(h_s - a_s)) * (e - e_home)
            ratings[h_id] += shift
            ratings[a_id] -= shift
        
        return total_rps / count if count > 0 else 0

    WARMING_ELO = 18935
    TRAIN_END = 21716
    data_array = df.iloc[:TRAIN_END][['h_id', 'a_id', 'home_score', 'away_score']].values.astype(int)
    res = minimize(
        objective_function, [20.0, 40.0, 90.0], 
        args=(data_array, len(all_teams), WARMING_ELO),
        bounds=[(5, 80), (0, 150), (10, 200)], method='L-BFGS-B'
        )
    
    return res.x


### 3.2 Burn-in Strategy (1872 – 2018)


A critical challenge in ELO-based modeling is the initialization bias. Initially all teams has an arbitrary rating of 1500 points, therefore the initial matches and ELO calculations are influentiated for their initial arbitrary values. To reduce this effect, we consider a burn-in strategy:

Stability Phase: The first several thousand matches (from 1872 to the 2018 FIFA World Cup) are processed solely to update the ratings. During this phase, the ratings "stabilize" as the system learns the relative strengths of the teams through historical results. For these matches we do not calculate the RPS. We calculate RPS and perform the optimization of parameters from 2018 FIFA World Cup till 2022 FIFA World Cup



In [7]:
def run_static_validation(df_all, train_params, start_eval_idx, end_eval_idx, large_visualization=True):
    k_base, hfa, draw_margin = train_params
    num_teams = len(all_teams)
    ratings = np.full(num_teams, 1500.0)
    rps_scores = []
    

    print(f"Validación: K={k_base:.1f}, HFA={hfa:.1f}, Margin={draw_margin:.1f}")
    print("-" * 85)
    df_window = df_all.iloc[:end_eval_idx]

    for i, row in df_window.iterrows():
        h_id, a_id = row['h_id'], row['a_id']
        h_s, a_s = int(row['home_score']), int(row['away_score'])
        
        # prediction
        delta = ratings[h_id] + hfa - ratings[a_id]
        p_away = 1 / (1 + 10 ** ((delta + draw_margin) / 400))
        p_home = 1 / (1 + 10 ** (-(delta - draw_margin) / 400))
        p_draw = max(0.0, 1.0 - p_home - p_away)
        probs = [p_home, p_draw, p_away]
        
        actual = 0 if h_s > a_s else 1 if h_s == a_s else 2
        r_h_prev, r_a_prev = ratings[h_id], ratings[a_id]

        if i >= start_eval_idx:
            # Calculate RPS
            p_cum = np.cumsum(probs)
            e_cum = [1.0 if actual == 0 else 0.0, 1.0 if actual <= 1 else 0.0]
            current_rps = ((p_cum[0]-e_cum[0])**2 + (p_cum[1]-e_cum[1])**2)/2.0
            rps_scores.append(current_rps)
            
            if large_visualization:
                status = "✅" if np.argmax(probs) == actual else "❌"
                print(f"[{row['date'].strftime('%Y-%m-%d')}] {row['home_team']:<20} {h_s}-{a_s:^3} {row['away_team']:>20} | {status}")
                print(f"  └─ Pred: L: {p_home:>6.1%} | E: {p_draw:>6.1%} | V: {p_away:>6.1%}")
                print(f"  └─ Elo: {r_h_prev:.0f} vs {r_a_prev:.0f} | RPS: {current_rps:.4f}")
                print("-" * 85)
            elif i % 5000 == 0:
                print(f"Procesando: {i}/{len(df_all)}...")

        # Update
        outcome = 1.0 if h_s > a_s else 0.5 if h_s == a_s else 0.0
        e_home = 1 / (1 + 10 ** (-delta / 400))
        k_mod = k_base * get_k_multiplier(h_s - a_s)
        
        shift = k_mod * (outcome - e_home)
        ratings[h_id] += shift
        ratings[a_id] -= shift

    return np.mean(rps_scores)

WARMING_ELO = 18935
TRAIN_END = 21716
    
df_train = df.iloc[:TRAIN_END]

## **Unit 4: Model Validation: Qatar 2022**
### 4.1 Backtesting & Out-of-Sample Performance
    
For testing the performance of our model with the optimal parameters obtained We execute a validation phase using the FIFA World Cup 2022 matches. This dataset was not included in the parameter calibration phase, serving as an "out-of-sample" test.    

During this phase:
Neutral Ground Assumption: We set the Home Field Advantage ($d_{home}$) to 0, as World Cup matches are played in neutral venues (with the exception of the host nation, which we generalize for model simplicity).
Iterative Evaluation: Ratings continue to update after each match, but we start recording the RPS to measure how well the model's probabilities align with the tournament's surprises and certainties.

## Training and validating model

In order to obtain the optimal values  of $k_{base}$, $d_{\text{home}}$ and $c$, we should minimize the RPS. For a robust result, we implement a burn-in period:

Phase 1: Burn-in: the first $N$ matches are used to obtain a brief approximation of the relative strength of the teams. Update the ratings without considering the RPS.

Phase 2: Scoring: after passing the first $N$ matches, we start to compute the RPS. This allows for optimizing parameters over quite reliable ranks for the teams.

This is important because, without the burn-in, the optimizer would pick a $K$ value that is artificially high just to "fix" the initial 1500 bias, leading to overfitting and poor predictive power in the validation set.

We consider the train set the matches prior to the FIFA World Cup 2022 and the burn-in period till the FIFA World Cup 2018.

Finally, for the World Cup matches, we set $d_{\text{home}}$ to $0$ because most teams do not have home-field advantage.

To validate the model, we compute the RPS for the matches in the FIFA World Cup 2022.






In [8]:
# Fixed parameters for World Cup validation
best_params = train_parameters(df.iloc[:TRAIN_END], len(all_teams),WARMING_ELO)    
best_params[1] = 0 # Setting HFA to 0 for neutral ground


VALIDATION_END = 21780

# Running validation on Qatar 2022
final_rps = run_static_validation(df,
                                  best_params,
                                  start_eval_idx=TRAIN_END,
                                  end_eval_idx=VALIDATION_END,
                                  large_visualization=True)
print(final_rps)

Validación: K=32.9, HFA=0.0, Margin=107.9
-------------------------------------------------------------------------------------
[2022-11-20] Qatar                0- 2               Ecuador | ❌
  └─ Pred: L:  49.1% | E:  27.9% | V:  23.0%
  └─ Elo: 1850 vs 1749 | RPS: 0.4165
-------------------------------------------------------------------------------------
[2022-11-21] United States        1- 1                 Wales | ❌
  └─ Pred: L:  63.9% | E:  22.1% | V:  14.0%
  └─ Elo: 1938 vs 1731 | RPS: 0.2138
-------------------------------------------------------------------------------------
[2022-11-21] Senegal              0- 2           Netherlands | ✅
  └─ Pred: L:  15.8% | E:  23.6% | V:  60.6%
  └─ Elo: 1775 vs 1958 | RPS: 0.0902
-------------------------------------------------------------------------------------
[2022-11-21] England              6- 2                  Iran | ❌
  └─ Pred: L:  24.7% | E:  28.5% | V:  46.8%
  └─ Elo: 1862 vs 1948 | RPS: 0.3926
--------------------------

In [9]:
def objective_function(params, data_array, num_teams, burn_in):
        k_base, hfa, draw_margin = params
        ratings = np.full(num_teams, 1500.0)
        total_rps = 0.0
        count = 0
    
        for i in range(len(data_array)):
            h_id, a_id, h_s, a_s = data_array[i]
            delta = ratings[h_id] + hfa - ratings[a_id]
        
            p_away = 1 / (1 + 10 ** ((delta + draw_margin) / 400))
            p_home = 1 / (1 + 10 ** (-(delta - draw_margin) / 400))
            p_draw = max(0.0, 1.0 - p_home - p_away)
        
            actual = 0 if h_s > a_s else 1 if h_s == a_s else 2
        
            if i >= burn_in:
                e_cum_0 = 1.0 if actual == 0 else 0.0
                e_cum_1 = 1.0 if actual <= 1 else 0.0
                total_rps += ((p_home - e_cum_0)**2 + (p_home + p_draw - e_cum_1)**2) / 2.0
                count += 1
        
            # Update ratings
            e = 1.0 if actual == 0 else 0.5 if actual == 1 else 0.0
            e_home = 1 / (1 + 10 ** (-delta / 400))
            shift = (k_base * get_k_multiplier(h_s - a_s)) * (e - e_home)
            ratings[h_id] += shift
            ratings[a_id] -= shift
        
        return total_rps / count if count > 0 else 0
data_array = df_train[['h_id', 'a_id', 'home_score', 'away_score']].values.astype(int)    


(21716, 4)
0.1713154737804116
0.17131123067690746


## Unit 5: Predictive Engine for 2026
### 5.1 Final Rating Consolidation

Before moving to the simulation phase, we consolidate the final ELO ratings for all 48 teams participating in the 2026 World Cup. These ratings, adjusted after the 2022 cycle, represent our "Ground Truth" for the starting strength of each nation.

### 5.2 Poisson-Based Simulation

While the ELO system provides probabilities for $W-D-L$, it does not inherently predict specific scores. To simulate the 2026 tournament, we implement a Monte Carlo approach using a Poisson Distribution
$$P(X = k) = \frac{\lambda^k \cdot e^{-\lambda}}{k!}$$
where: $P(X = k)$: probability of $k$ goals; $\lambda$, the number of expected goals in the match.

This model is adjusted with the ELO for each team, then the probability of a result of i-j is:
$$P(X = i, Y=j) = \frac{\lambda_x^i \cdot e^{-\lambda_x}}{i!}\frac{\lambda_y^j \cdot e^{-\lambda_y}}{j!}$$
where $\lambda_x$ and $\lambda_y$ are the expected goals per team adjusted by ELO
$$
\lambda_x=\lambda \frac{1}{10^{(ELO_x-ELO_y ) / 400} + 1},\ \   \ \lambda_y=\lambda \frac{1}{10^{(ELO_y-ELO_x ) / 400} + 1}  
$$
where $\lambda$ is the average goals.


In [10]:
def simulate_match(elo_a, elo_b,rng , avg_goals=1.3):
    # Calculate expected goals (lambda) based on Elo difference
    diff = elo_a - elo_b
    lambda_a = avg_goals * (1/(1+10**(-diff / 400)))
    lambda_b = avg_goals * (1/(1+10**(diff / 400)))
    
    # Generate random goals
    return rng.poisson(lambda_a,1), rng.poisson(lambda_b,1)

def simulate_match_montecarlo(elo_a, elo_b,rng ,n_times, avg_goals=1.3):
    max_goals = 8
    diff = elo_a - elo_b
    lambda_a = avg_goals * (1 / (1 + 10**(-diff / 400)))
    lambda_b = avg_goals * (1 / (1 + 10**(diff / 400)))
    
    goals_a = rng.poisson(lambda_a, n_times)
    goals_b = rng.poisson(lambda_b, n_times)
    
    A, _, _ = np.histogram2d(
        goals_a, goals_b, 
        bins=(np.arange(max_goals + 1), np.arange(max_goals + 1))
    )
    probs = A/A.sum()
    e_b = np.ones(max_goals) @ probs @ np.arange(max_goals)
    e_a = np.arange(max_goals) @ probs @ np.ones(max_goals)
    
    a_score, b_score = np.unravel_index(np.argmax(probs), probs.shape)

    return probs, (a_score, b_score), np.max(probs), (e_a,e_b)


